---
title: "California Mother Lode v3.1: data exploration of new layers"
subtitle: "Magnetic-field derivatives, fine-grained lithology, deposit-type fields"
format:
  html:
    toc: true
    code-fold: show
execute:
  warning: false
---

This notebook tours the new data layers added in the Mother Lode v3.1
increment. The v3 data tour at
[`data_exploration.qmd`](data_exploration.qmd) covers the original
layers (MRDS, SGMC, NGDB, magnetic, gravity, DEM, Sentinel-2). This
companion covers only what's new.

In [ ]:
#| label: setup
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from ai_minerals.regions.motherlode import MOTHERLODE
from ai_minerals.data._common import DATA_RAW

## 1. Magnetic-field derivatives

### Why magnetic data helps locate gold

The Earth's crust is not magnetically uniform. Iron-bearing minerals,
mostly magnetite with smaller contributions from pyrrhotite and
hematite, align weakly with the planet's field. Rocks containing more
of these minerals produce a slightly stronger local field; rocks
containing fewer produce a weaker one. An airborne magnetometer
measures this variation across the surface; subtracting the smooth
global background leaves the *residual* field, which is essentially a
map of magnetic-mineral distribution in the upper crust.

Orogenic gold itself has no magnetic signature. The host rocks around
it do. The Mother Lode gold sits inside a north-south greenstone belt
of slate, schist, greenstone, and serpentinite, and many of these
rocks are magnetically distinct from the granitic Sierra Nevada
batholith east of the belt. The boundary between the two terranes is
the same boundary that controls the gold-bearing fault system. So a
model that uses the magnetic field is, in effect, learning the
structural geometry of the belt without ever being told the
geologist's labels for individual rock units.

### What the four derivatives compute

The raw residual field gives one number per cell: how strongly the
local rock is magnetized. *Derivatives* describe how that field
changes across space. Each highlights a different geometric property
of the underlying rocks. v3 used only the raw field; v3.1 adds the
four standard derivatives.

- **1VD (first vertical derivative).** How fast the magnetic field
  changes with depth. Computed by multiplying each spatial-frequency
  component of the field by its wavenumber. Geological intuition:
  highlights shallow magnetic sources and suppresses deep ones,
  sharpening near-surface boundaries.
- **HGM (horizontal gradient magnitude).** How fast the field
  changes laterally, in any direction. Computed by taking the
  east and north spatial derivatives and combining them via
  Pythagoras. Geological intuition: peaks over vertical or
  near-vertical contacts between rocks of different magnetic
  susceptibility; an edge map.
- **Analytic signal.** Combines 1VD and HGM into a single value via
  Pythagoras applied to their magnitudes. Useful because it peaks
  directly above a magnetic body regardless of orientation, so
  edges can be located without knowing the ambient field direction.
- **Tilt derivative.** Roughly `arctan(1VD / HGM)`. The arctan
  squashes large values toward ±π/2 and small ones toward zero, so
  weak anomalies become visually as prominent as strong ones.
  Useful when faint-but-real anomalies need to compete with loud
  edges of the survey area.

All four are standard features in the published MPM literature
(Blakely 1995, *Potential Theory in Gravity and Magnetic
Applications* ch. 7-12; Verduzco et al. 2004 "The tilt derivative
makes magnetic interpretation easier"). Implementation:
[`src/ai_minerals/data/magnetic_derivatives.py`].

In [ ]:
#| label: mag-derivs-grid
fig, axes = plt.subplots(2, 2, figsize=(13, 12))
panels = [
    ("magnetic_1vd_motherlode.tif",            "1VD (1st vertical derivative)"),
    ("magnetic_hgm_motherlode.tif",            "HGM (horizontal gradient)"),
    ("magnetic_analytic_signal_motherlode.tif","Analytic signal"),
    ("magnetic_tilt_motherlode.tif",           "Tilt"),
]
for ax, (name, title) in zip(axes.ravel(), panels):
    p = DATA_RAW / "gsc_geophysics" / name
    with rasterio.open(p) as src:
        arr = src.read(1)
        bounds = src.bounds
        nodata = src.nodata
    show = np.where(np.isfinite(arr) & (arr != nodata if nodata is not None else True), arr, np.nan)
    extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
    im = ax.imshow(
        show, extent=extent, origin="upper", cmap="RdBu_r",
        vmin=np.nanpercentile(show, 5),
        vmax=np.nanpercentile(show, 95),
    )
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

**Interpretation.** All four derivatives sharpen the
N-S-foothills-belt magnetic high relative to the raw field. 1VD and
HGM emphasize the gradient (the boundary between magnetic-positive
greenstone-belt rocks and the magnetic-low Sierra Nevada batholith).
The analytic signal is nearly identical to HGM in the foothills
because the field is mostly horizontal-gradient-dominated. Tilt
flattens the amplitude so that weak anomalies in the foothills read
at the same brightness as the strongest ones, which is what makes it
useful as a model feature: the model gets to see edge geometry
without being dominated by amplitude.

## 2. SGMC fine-grained MAJOR1/2/3 lithology

v3 used SGMC's `GENERALIZED_LITH` controlled-vocabulary field
(33 classes) as the lithology one-hot. v3.1 adds the `MAJOR1`,
`MAJOR2`, `MAJOR3` fields, which carry hundreds of distinct named
lithologies (Mariposa Slate, Calaveras Schist, Granodiorite, etc.).
Each is factorized to an integer code per region and one-hot
encoded for the top-10 most common values.

In [ ]:
#| label: major1-distribution
import fiona
gdb = "/home/sky/src/learning/ai-minerals/data/raw/sgmc/USGS_SGMC_Geodatabase/USGS_StateGeologicMapCompilation_ver1.1.gdb"
ca_aoi = MOTHERLODE.aoi.bbox
from collections import Counter
def _state_filter(layer="SGMC_Geology", state="CA"):
    rows = []
    with fiona.open(gdb, layer=layer) as src:
        for feat in src:
            if feat["properties"].get("STATE") == state:
                rows.append(feat["properties"])
    return rows

ca = _state_filter()
print(f"CA SGMC records: {len(ca):,}")

major1_counts = Counter(p.get("MAJOR1", "") for p in ca)
print("\nTop 20 MAJOR1 values in CA:")
for v, n in major1_counts.most_common(20):
    print(f"  {n:>5}  {v[:60]}")

In [ ]:
#| label: major1-mother-lode
# Same but filtered to the Mother Lode AOI bbox.
from shapely.geometry import box, Point
import geopandas as gpd
mother_box = box(*MOTHERLODE.aoi.bbox)
geo_path = DATA_RAW / "sgmc" / "sgmc_geology_motherlode.gpkg"
mlgeo = gpd.read_file(geo_path)
print(f"Mother Lode SGMC polygons: {len(mlgeo):,}")
print("\nTop 20 MAJOR1 values in Mother Lode AOI:")
for v, n in mlgeo.get("MAJOR1", pd.Series([])).value_counts().head(20).items():
    print(f"  {n:>5}  {v[:60] if v else '(blank)'}")

**Interpretation.** The Mother Lode AOI MAJOR1 distribution is
dominated by Slate, Schist, Greenstone, Serpentinite, Tonalite,
Granodiorite, and Marble. These are the rock types that host the
Mother Lode Belt's orogenic gold (slate + schist of the Mariposa-
Calaveras sequences) and the surrounding terrain (granitic Sierra
Nevada batholith). The v3.1 model gets to learn these specific
unit-name signals on top of the GENERALIZED_LITH coarse classes.

## 3. MRDS deposit-type and operation-type fields

v3 commodity-filtered MRDS records to "anything with Au or gold."
That commodity filter mixes orogenic Au (lode, vein-hosted) with
placer Au (alluvial, stream-bed). v3.1 uses the `oper_type` and
`dep_type` fields to filter out placer records before they enter the
positive label set.

In [ ]:
#| label: oper-type-distribution
mrds = gpd.read_file(DATA_RAW / "mrds" / "mrds_motherlode.gpkg")
au = mrds[mrds["commodity"].str.contains("au|gold", case=False, regex=True, na=False)]
print(f"Au-bearing records in Mother Lode AOI: {len(au):,}")
print("\noper_type distribution among Au records:")
for v, n in au["oper_type"].fillna("(blank)").value_counts().head(10).items():
    print(f"  {n:>6}  {v}")

In [ ]:
#| label: dep-type-distribution
print("dep_type distribution among Au records (top 15 non-blank):")
counts = au[au["dep_type"].notna() & (au["dep_type"] != "")]["dep_type"].value_counts().head(15)
for v, n in counts.items():
    print(f"  {n:>6}  {v[:80]}")

**Interpretation.** About 15% of MRDS Au records in Mother Lode are
flagged as placer (1,981 / 13,305 by `oper_type == "Placer"`), and
another ~1,000 carry placer-style language in `dep_type` ("Placer",
"Stream Placer", "Alluvial"). These are alluvial gold sites, not
in-place orogenic-Au mineralization. v3.1's adapter excludes them
from the orogenic-Au positive set, leaving them in the
any-mineral-occurrence exclusion mask but out of the binary label
the model trains against.

## 4. Notes on what's not changed from v3

The data layers themselves (raw magnetic, gravity, NGDB, SGMC, MRDS,
DEM, Sentinel-2) are unchanged from v3. v3.1 is feature engineering
plus label cleanup, not new fetches. The v3 data tour at
[`data_exploration.qmd`](data_exploration.qmd) remains the canonical
reference for the underlying datasets.